# <font color="red">**Higher-Dimensional Optical Data Visualization Challenge**</font>

## Composition → spectrum → measurement context

### The problem

Optical libraries are difficult to visualize because every sample has a composition,
a full spectrum, a measurement geometry, and sometimes an elapsed-time history. A
collection of separate plots makes comparisons slow, but forcing everything into one
crowded plot can hide the spectral evidence.

> **Can we make all of those variables easier to explore without hiding the spectra
> or compressing the experiment into a single unexplained score?**

### The approach

Like the example EELS hackathon notebook, this notebook follows one linear pattern:

1. inspect the data and its dimensions;
2. run one complete, deliberately simple worked example;
3. modify that baseline in a clearly marked challenge cell;
4. test whether a new user can answer the stated scientific question.

### Choose one challenge

1. **Challenge 1 — Quaternary System:** connect FA/Cs/Rb/DMA composition and I/Br variation to
   Top PL, Bottom PL, and absorbance.
2. **Challenge 2 — RP-Additive System:** connect 3AMP/FAPbI3 host composition and nominal MACl
   level to Top/Bottom PL over 0–380 minutes.

### Your final result

- one visualization that answers a specific scientific question;
- the full spectral evidence remains reachable or visible;
- one sentence explaining what became easier to see and one possible misinterpretation.

The worked examples are deliberately simple baselines not
the expected final answer.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbarakati/athena_camm_hackathon/blob/k4my4r/docs/day_19_02102026/Higher_Dimensional_Optical_Data_Hackathon.ipynb)

<h2><font color="purple">1.</font> Download and Load the Data</h2>

Run the next cells from top to bottom. One public Google Drive folder contains both
challenge folders. The download cell is safe to rerun: it reuses a verified local copy
and only downloads again when files are missing or changed.


In [1]:
# @title 1. Install and import the small analysis stack
#%pip -q install "gdown==6.4.1" "pyarrow>=14" "plotly>=5.22"

from html import escape
from pathlib import Path
import hashlib
import shutil
import sys

import gdown
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import FileLink, HTML, Markdown, display


In [ ]:
# # @title 2. Download both challenge folders from Google Drive
# DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/16YfPjFdZHgW3hkC0QBLrCS9um2C6aK5q?usp=drive_link"
# IN_COLAB = "google.colab" in sys.modules
# DATA_ROOT = Path(
#     "/content/higher_dimensional_optical_data"
#     if IN_COLAB else "./.higher_dimensional_optical_data"
# ).resolve()
# STAGING_ROOT = DATA_ROOT.with_name(DATA_ROOT.name + "_download")

# REMOTE_QUATERNARY_FOLDER = "01_hochan_quaternary_data"
# EXPECTED_MANIFEST_HASHES = {
#     REMOTE_QUATERNARY_FOLDER: "6c4f5329411f7897b16578ec3f029f092ca05eb99a884a9a622b1ffabf24ff8e",
#     "02_rp_additive_data": "0060b641f417179122562eefb6d5acee527b4927796fd4a595631b683fded38c",
# }

# def file_sha256(path):
#     digest = hashlib.sha256()
#     with Path(path).open("rb") as handle:
#         for chunk in iter(lambda: handle.read(1024 * 1024), b""):
#             digest.update(chunk)
#     return digest.hexdigest()

# def folder_errors(root):
#     errors = []
#     for folder_name, expected_manifest_hash in EXPECTED_MANIFEST_HASHES.items():
#         folder = root / folder_name
#         checksum_path = folder / "SHA256SUMS.txt"
#         if not checksum_path.is_file():
#             errors.append(f"missing: {folder_name}/SHA256SUMS.txt")
#             continue
#         if file_sha256(checksum_path) != expected_manifest_hash:
#             errors.append(f"unexpected checksum manifest: {folder_name}")
#             continue
#         for line in checksum_path.read_text(encoding="utf-8").strip().splitlines():
#             expected, relative = line.split("  ", 1)
#             path = folder / relative
#             if not path.is_file():
#                 errors.append(f"missing: {folder_name}/{relative}")
#             elif file_sha256(path) != expected:
#                 errors.append(f"checksum mismatch: {folder_name}/{relative}")
#     return errors

# def locate_download_root(root):
#     candidates = [root] + sorted(
#         (path for path in root.rglob("*") if path.is_dir()),
#         key=lambda path: len(path.parts),
#     )
#     matches = [
#         path for path in candidates
#         if (path / REMOTE_QUATERNARY_FOLDER).is_dir()
#         and (path / "02_rp_additive_data").is_dir()
#     ]
#     if len(matches) != 1:
#         raise RuntimeError(
#             "Expected one downloaded parent containing both challenge folders; "
#             f"found {len(matches)}."
#         )
#     return matches[0]

# errors = folder_errors(DATA_ROOT)
# if errors:
#     if STAGING_ROOT.exists():
#         shutil.rmtree(STAGING_ROOT)
#     STAGING_ROOT.mkdir(parents=True)
#     print("Downloading both challenge datasets from Google Drive ...")
#     try:
#         downloaded = gdown.download_folder(
#             url=DRIVE_FOLDER_URL,
#             output=str(STAGING_ROOT),
#             quiet=False,
#             use_cookies=False,
#         )
#         if not downloaded:
#             raise RuntimeError("Google Drive returned no files.")
#         downloaded_root = locate_download_root(STAGING_ROOT)
#         errors = folder_errors(downloaded_root)
#         if errors:
#             raise RuntimeError("Downloaded data failed verification:\n- " + "\n- ".join(errors))

#         if DATA_ROOT.exists():
#             shutil.rmtree(DATA_ROOT)
#         if downloaded_root == STAGING_ROOT:
#             STAGING_ROOT.rename(DATA_ROOT)
#         else:
#             shutil.move(str(downloaded_root), str(DATA_ROOT))
#             shutil.rmtree(STAGING_ROOT, ignore_errors=True)
#     except Exception as exc:
#         shutil.rmtree(STAGING_ROOT, ignore_errors=True)
#         raise RuntimeError(
#             "Data setup failed. Confirm that the shared parent folder and its contents "
#             "are set to 'Anyone with the link - Viewer', then rerun this cell."
#         ) from exc
#     print("Download complete; every file passed SHA-256 verification.")
# else:
#     print("Verified data already present; no download needed.")

# QUATERNARY_DIR = DATA_ROOT / REMOTE_QUATERNARY_FOLDER
# RP_DIR = DATA_ROOT / "02_rp_additive_data"
# print("Challenge 1 — Quaternary System: ready")
# print("Challenge 2 — RP-Additive System: ready")


Retrieving folder contents


Retrieving folder 1lKab_j038FiNxPvNzHtnMUv_U9es1gN7 01_hochan_quaternary_data
Processing file 1Z20at8pdv7XnxWKYNDkNUDtzwHMHnrOu composition_and_recipe.csv
Processing file 1lIYfpqZTv7B1rU9QVv3zWyK7QP_xaZn6 measurement_manifest.csv
Processing file 16JOB1gy7TrfBvBp68hgJR4FcQjw_c5lZ optical_read_qc_summary.csv
Processing file 1VIJhV0YhdqfpmMIZxljMzWyOwWc1pYnz optical_spectra_long.parquet
Processing file 19Gmib_Jf0RhhJCsw_R9oxpNxfQmQG-9g README_DATA.md
Processing file 1h-cmt1usy-jDRkRNU3VqWZtR_Xo7DFOW SHA256SUMS.txt
Retrieving folder 1XCDHKsa7-_pmEmpH1EPpArxUlf54HvIH 02_rp_additive_data
Processing file 1bTvZUNQbc7FtEtNSXDBBGGoe000OPvcz composition_and_recipe.csv
Processing file 1qJ8wxyUdn_1rSESAV8bm_LJcsIBK0QJV metadata_features.parquet
Processing file 1Rz8SY732Cbf7oONnmumZcxE8Dt2a1lmt README_DATA.md
Processing file 1Rl82CZGXM248CoWYgy9KmJvBzspEg5rm SHA256SUMS.txt
Processing file 1nGx6ng2If1IlFK_l4lSGXlLzbgbWR1Lc spectra_wide.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1Z20at8pdv7XnxWKYNDkNUDtzwHMHnrOu
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/composition_and_recipe.csv
100%|██████████| 20.5k/20.5k [00:00<00:00, 8.37MB/s]
Downloading...
From: https://drive.google.com/uc?id=1lIYfpqZTv7B1rU9QVv3zWyK7QP_xaZn6
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/measurement_manifest.csv
100%|██████████| 692/692 [00:00<00:00, 1.26MB/s]
Downloading...
From: https://drive.google.com/uc?id=16JOB1gy7TrfBvBp68hgJR4FcQjw_c5lZ
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/optical_read_qc_summary.csv
100%|██████████| 630/630 [00:00<00:00, 1.52MB/s]
Downloading...
From: https://drive.google.com/uc?id=1VIJhV0YhdqfpmMIZxljMzWyOwWc1pYnz
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/

Download complete; every file passed SHA-256 verification.
Challenge 1 — Quaternary System: ready
Challenge 2 — RP-Additive System: ready



Download completed


In [2]:
# @title 2. Use the local data folders (no download)
from pathlib import Path
import hashlib

# The folder that holds the two data folders.
# The notebook is in the same folder as the data, so we use the current folder.
# If you get a "missing" error, write the full path instead, for example:
# DATA_ROOT = Path("/Users/borisslautin/Documents/GitHub/athena_camm_hackathon/docs/day_19_02102026")
DATA_ROOT = Path.cwd().resolve()

# The two data folders (the next cells use these two names)
QUATERNARY_DIR = DATA_ROOT / "01_hochan_quaternary_data"
RP_DIR = DATA_ROOT / "02_rp_additive_data"

# Expected fingerprint of the checksum list in each folder
EXPECTED_MANIFEST_HASHES = {
    QUATERNARY_DIR: "6c4f5329411f7897b16578ec3f029f092ca05eb99a884a9a622b1ffabf24ff8e",
    RP_DIR: "0060b641f417179122562eefb6d5acee527b4927796fd4a595631b683fded38c",
}


def file_sha256(path):
    """Compute the SHA-256 fingerprint of one file."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Check that every file is present and unchanged
problems = []
for folder, expected_list_hash in EXPECTED_MANIFEST_HASHES.items():
    checksum_list = folder / "SHA256SUMS.txt"
    if not checksum_list.is_file():
        problems.append(f"missing: {checksum_list}")
        continue
    if file_sha256(checksum_list) != expected_list_hash:
        problems.append(f"unexpected checksum list: {checksum_list}")
    # Each line in SHA256SUMS.txt is: <fingerprint>  <file name>
    for line in checksum_list.read_text(encoding="utf-8").strip().splitlines():
        expected_hash, file_name = line.split("  ", 1)
        path = folder / file_name
        if not path.is_file():
            problems.append(f"missing: {path}")
        elif file_sha256(path) != expected_hash:
            problems.append(f"changed or damaged: {path}")

# Stop here if something is wrong, and show what is wrong
if problems:
    raise FileNotFoundError("Local data check failed:\n- " + "\n- ".join(problems))

print("Using local data in:", DATA_ROOT)
print("Challenge 1 — Quaternary System: ready")
print("Challenge 2 — RP-Additive System: ready")

Using local data in: /Users/borisslautin/Documents/GitHub/athena_camm_hackathon/docs/day_19_02102026
Challenge 1 — Quaternary System: ready
Challenge 2 — RP-Additive System: ready


In [3]:
# @title 3. Load the prepared tables
QUATERNARY_COMPOSITION_PATH = QUATERNARY_DIR / "composition_and_recipe.csv"
RP_COMPOSITION_PATH = RP_DIR / "composition_and_recipe.csv"

quaternary_composition = pd.read_csv(QUATERNARY_COMPOSITION_PATH)
quaternary_optical = pd.read_parquet(QUATERNARY_DIR / "optical_spectra_long.parquet")
quaternary_manifest = pd.read_csv(QUATERNARY_DIR / "measurement_manifest.csv")

rp_composition = pd.read_csv(RP_COMPOSITION_PATH)
rp_features = pd.read_parquet(RP_DIR / "metadata_features.parquet")
rp_spectra = pd.read_parquet(RP_DIR / "spectra_wide.parquet")

rp_wavelength_columns = sorted(
    (column for column in rp_spectra if column.startswith("wl_")),
    key=lambda column: float(column.split("_", 1)[1]),
)
rp_wavelengths = np.asarray([
    float(column.split("_", 1)[1]) for column in rp_wavelength_columns
])
rp_spectra_by_id = rp_spectra.set_index("spectrum_id")


In [4]:
# @title 4. Run compact scientific-structure checks
assert len(quaternary_composition) == quaternary_composition["well"].nunique() == 96
assert set(quaternary_composition["well"]) == set(quaternary_optical["well"])
assert set(quaternary_optical["geometry"]) == {"Top", "Bottom", "Absorbance"}

a_site = ["FA_fraction", "Cs_fraction", "Rb_fraction", "DMA_fraction"]
x_site = ["I_fraction", "Br_fraction", "Cl_fraction"]
assert np.allclose(quaternary_composition[a_site].sum(axis=1), 1.0)
assert np.allclose(quaternary_composition[x_site].sum(axis=1), 1.0)

assert len(rp_composition) == rp_composition["well"].nunique() == 96
assert set(rp_composition["well"]) == set(rp_features["well"])
assert len(rp_features) == rp_features["spectrum_id"].nunique() == 3840
assert set(rp_features["spectrum_id"]) == set(rp_spectra["spectrum_id"])
assert np.allclose(
    rp_composition["spacer_host_fraction"]
    + rp_composition["FAPbI3_host_fraction"],
    1.0,
)

coverage = pd.DataFrame([
    {
        "challenge": "Challenge 1 — Quaternary System",
        "wells": quaternary_composition["well"].nunique(),
        "spectra": quaternary_optical.groupby(["well", "geometry"]).ngroups,
        "times": "none",
        "measurements": "Top PL, Bottom PL, absorbance",
    },
    {
        "challenge": "Challenge 2 — RP-Additive System",
        "wells": rp_composition["well"].nunique(),
        "spectra": len(rp_features),
        "times": rp_features["time_min"].nunique(),
        "measurements": "Top PL, Bottom PL",
    },
])
display(Markdown("### Data ready — all integrity and structure checks passed"))
display(coverage)


### Data ready — all integrity and structure checks passed

,challenge,wells,spectra,times,measurements
0,Challenge 1 — Quaternary System,96,288,none,"Top PL, Bottom PL, absorbance"
1,Challenge 2 — RP-Additive System,96,3840,20,"Top PL, Bottom PL"


## What is “high-dimensional” here?

| Challenge | Composition variables | Measurement variables | Spectral variable |
|---|---|---|---|
| Quaternary System | FA, Cs, Rb, DMA; I/Br slice | Top PL, Bottom PL, absorbance | wavelength |
| RP-Additive System | 3AMP/FAPbI3 host fraction; nominal MACl level | time; Top/Bottom PL | wavelength |

Two guardrails matter throughout:

- In the Quaternary System, **Top and Bottom are collection geometries**, not time
  points or replicates.
- In the RP-Additive System, MACl percentage is a **nominal design level with
  undocumented basis**,
  not a third closed-composition fraction.

PL peaks and spectral summaries are optical observations—not structural phase labels.

### Possible visualization strategies

There is no single required chart type. Useful solution families include:

- **small multiples:** reserve panels for composition slices, geometry, or time;
- **overview + detail:** use a composition map to navigate to complete spectra;
- **interactive controls:** let the user select a well, time, geometry, or metric;
- **change encodings:** show Top − Bottom or final − initial values with a shared scale;
- **spectral-first views:** use heatmaps, trajectories, or compact spectral glyphs.

The worked examples below use **overview + detail**. They are starting points, not the
intended final answer. Figures are stacked at full width by default so labels, legends,
and color scales stay readable in Colab.


In [5]:
# Inspect or download the two one-row-per-well composition tables
display(Markdown("#### Challenge 1 — Quaternary System composition and recipe"))
display(quaternary_composition.head(6))
display(FileLink(
    str(QUATERNARY_COMPOSITION_PATH), result_html_prefix="Download full CSV: "
))

display(Markdown("#### Challenge 2 — RP-Additive System composition and recipe"))
display(rp_composition.head(6))
display(FileLink(str(RP_COMPOSITION_PATH), result_html_prefix="Download full CSV: "))


#### Challenge 1 — Quaternary System composition and recipe

,well,row,col,FA_fraction,Cs_fraction,Rb_fraction,DMA_fraction,I_fraction,Br_fraction,Cl_fraction,...,Br_design_percent,total_volume_uL,FAPbI3_volume_uL,FAPbBr3_volume_uL,RbPbI3_volume_uL,DMAPbI3_volume_uL,CsPbI3_volume_uL,CsPbI05Br05_volume_uL,composition_label,composition_basis
0,A1,A,1,0.900,0.1,0.0,0.000,1.0000,0.0000,0.0,...,0.000000,100.0,90.0,0.0,0.0,0.0,10.0,0.0,FA 0.900 | Cs 0.100 | Rb 0.000 | DMA 0.000 || ...,nominal_equal_molar_stock_volume
1,A2,A,2,0.875,0.1,0.0,0.025,1.0000,0.0000,0.0,...,0.000000,100.0,87.5,0.0,0.0,2.5,10.0,0.0,FA 0.875 | Cs 0.100 | Rb 0.000 | DMA 0.025 || ...,nominal_equal_molar_stock_volume
2,A3,A,3,0.850,0.1,0.0,0.050,1.0000,0.0000,0.0,...,0.000000,100.0,85.0,0.0,0.0,5.0,10.0,0.0,FA 0.850 | Cs 0.100 | Rb 0.000 | DMA 0.050 || ...,nominal_equal_molar_stock_volume
3,A4,A,4,0.800,0.1,0.0,0.100,1.0000,0.0000,0.0,...,0.000000,100.0,80.0,0.0,0.0,10.0,10.0,0.0,FA 0.800 | Cs 0.100 | Rb 0.000 | DMA 0.100 || ...,nominal_equal_molar_stock_volume
4,A5,A,5,0.900,0.1,0.0,0.000,0.8335,0.1665,0.0,...,16.666667,100.0,75.0,15.0,0.0,0.0,6.7,3.3,FA 0.900 | Cs 0.100 | Rb 0.000 | DMA 0.000 || ...,nominal_equal_molar_stock_volume
5,A6,A,6,0.875,0.1,0.0,0.025,0.8335,0.1665,0.0,...,16.666667,100.0,72.5,15.0,0.0,2.5,6.7,3.3,FA 0.875 | Cs 0.100 | Rb 0.000 | DMA 0.025 || ...,nominal_equal_molar_stock_volume


/Users/borisslautin/Documents/GitHub/athena_camm_hackathon/docs/day_19_02102026/01_hochan_quaternary_data/composition_and_recipe.csv

#### Challenge 2 — RP-Additive System composition and recipe

,well,row,col,library,base_spacer,additive,spacer_host_fraction,FAPbI3_host_fraction,spacer_percent,FAPbI3_percent,additive_percent,spacer_stock_volume_uL,FAPbI3_stock_volume_uL,additive_stock_volume_uL,host_total_uL,total_mix_volume_uL,composition_label,composition_basis
0,A1,A,1,3AMP-FA with MACl,3AMP,MACl,1.000000,0.000000,100.000000,0.000000,0.0,50.00,0.00,0.0,50.0,50.0,3AMP:1.00 FA:0.00 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
1,A2,A,2,3AMP-FA with MACl,3AMP,MACl,0.909091,0.090909,90.909091,9.090909,0.0,45.45,4.55,0.0,50.0,50.0,3AMP:0.91 FA:0.09 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
2,A3,A,3,3AMP-FA with MACl,3AMP,MACl,0.818182,0.181818,81.818182,18.181818,0.0,40.91,9.09,0.0,50.0,50.0,3AMP:0.82 FA:0.18 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
3,A4,A,4,3AMP-FA with MACl,3AMP,MACl,0.727273,0.272727,72.727273,27.272727,0.0,36.36,13.64,0.0,50.0,50.0,3AMP:0.73 FA:0.27 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
4,A5,A,5,3AMP-FA with MACl,3AMP,MACl,0.636364,0.363636,63.636364,36.363636,0.0,31.82,18.18,0.0,50.0,50.0,3AMP:0.64 FA:0.36 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
5,A6,A,6,3AMP-FA with MACl,3AMP,MACl,0.545455,0.454545,54.545455,45.454545,0.0,27.27,22.73,0.0,50.0,50.0,3AMP:0.55 FA:0.45 MACl:0%,host_fraction_from_nominal_stock_volume;additi...


/Users/borisslautin/Documents/GitHub/athena_camm_hackathon/docs/day_19_02102026/02_rp_additive_data/composition_and_recipe.csv

In [6]:
# Responsive display/export helpers used by both worked examples.
# One column is the readable Colab default; use columns=2 only on a wide display.
def show_figures(figures, columns=1):
    if columns not in (1, 2):
        raise ValueError("columns must be 1 or 2")
    blocks = [
        "<div style='min-width:0;width:100%;overflow-x:auto'>"
        + pio.to_html(
            figure,
            include_plotlyjs="cdn" if index == 0 else False,
            full_html=False,
            config={"responsive": True, "displaylogo": False},
        )
        + "</div>"
        for index, figure in enumerate(figures)
    ]
    grid_class = f"hd-figure-grid-{columns}"
    display(HTML(
        f"<style>.{grid_class}{{display:grid;grid-template-columns:repeat("
        f"{columns},minmax(0,1fr));gap:28px;align-items:start;width:100%;"
        "max-width:1200px;margin:0 auto}}"
        f"@media(max-width:1100px){{.{grid_class}{{grid-template-columns:1fr}}}}"
        "</style>"
        f"<div class='{grid_class}'>" + "".join(blocks) + "</div>"
    ))

def export_figure_grid(figures, path, title, columns=1):
    if columns not in (1, 2):
        raise ValueError("columns must be 1 or 2")
    blocks = [
        pio.to_html(
            figure,
            include_plotlyjs=True if index == 0 else False,
            full_html=False,
            config={"responsive": True, "displaylogo": False},
        )
        for index, figure in enumerate(figures)
    ]
    document = (
        "<!doctype html><meta charset='utf-8'><title>" + escape(title) + "</title>"
        "<style>body{font-family:Arial;margin:24px;color:#172b4d}"
        ".grid{display:grid;gap:28px;max-width:1200px;margin:0 auto}"
        f".grid{{grid-template-columns:repeat({columns},minmax(0,1fr))}}"
        "@media(max-width:1100px){.grid{grid-template-columns:1fr}}"
        ".grid>div{min-width:0;width:100%;overflow-x:auto}</style>"
        "<h1>" + escape(title) + "</h1><div class='grid'>"
        + "".join("<div>" + block + "</div>" for block in blocks)
        + "</div>"
    )
    Path(path).write_text(document, encoding="utf-8")
    return Path(path)


<h2><font color="purple">2.</font> Explore the Quaternary System</h2>

### Problem 1

> **Where do Top and Bottom PL disagree across composition, and what additional
> context does absorbance provide?**

The A-site fractions **FA + Cs + Rb + DMA = 1**. The X-site fractions
**I + Br + Cl = 1** separately. Never normalize all seven fractions together.

A conventional ternary diagram is not enough because the A-site contains four
components and the plate also varies halide composition. The baseline solution uses
**small-multiple composition slices** for overview and a separate **full-spectrum
detail view** for evidence.

We will first create a small well-level feature table. These scalar features are
navigation aids; the full spectra remain the evidence.


In [7]:
# Summarize each Quaternary System PL spectrum without changing the source data
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def summarize_pl(group):
    ordered = group.sort_values("wavelength_nm")
    wavelength = ordered["wavelength_nm"].to_numpy(float)
    raw = ordered["optical_signal_instrument_units"].to_numpy(float)
    signal = np.clip(raw - np.nanpercentile(raw, 5), 0, None)
    area = float(integrate(signal, wavelength))
    peak = float(wavelength[int(np.nanargmax(signal))])
    centroid = float(np.sum(wavelength * signal) / signal.sum())
    return pd.Series({
        "dominant_peak_nm": peak,
        "centroid_nm": centroid,
        "log10_integrated_pl": np.log10(area),
    })

quaternary_pl_features = (
    quaternary_optical[quaternary_optical["modality"].eq("PL")]
    .groupby(["well", "geometry"], sort=False)[
        ["wavelength_nm", "optical_signal_instrument_units"]
    ]
    .apply(summarize_pl)
    .reset_index()
)


In [8]:
# Assemble one row per well for composition-space visualization
pl_wide = quaternary_pl_features.pivot(
    index="well", columns="geometry",
    values=["dominant_peak_nm", "centroid_nm", "log10_integrated_pl"],
)
pl_wide.columns = [
    f"{geometry}_{metric}" for metric, geometry in pl_wide.columns
]
pl_wide = pl_wide.reset_index()

absorbance_points = (
    quaternary_optical[
        quaternary_optical["modality"].eq("Absorbance")
        & quaternary_optical["wavelength_nm"].isin([450, 650, 750])
    ]
    .pivot(index="well", columns="wavelength_nm",
           values="optical_signal_instrument_units")
    .rename(columns=lambda value: f"Abs_signal_{int(value)}nm")
    .reset_index()
)

quaternary_features = (
    quaternary_composition.merge(pl_wide, on="well", validate="one_to_one")
    .merge(absorbance_points, on="well", validate="one_to_one")
)
quaternary_features["Top_minus_Bottom_peak_nm"] = (
    quaternary_features["Top_dominant_peak_nm"]
    - quaternary_features["Bottom_dominant_peak_nm"]
)
quaternary_features["Top_minus_Bottom_centroid_nm"] = (
    quaternary_features["Top_centroid_nm"]
    - quaternary_features["Bottom_centroid_nm"]
)
display(quaternary_features.head())


,well,row,col,FA_fraction,Cs_fraction,Rb_fraction,DMA_fraction,I_fraction,Br_fraction,Cl_fraction,...,Top_dominant_peak_nm,Bottom_centroid_nm,Top_centroid_nm,Bottom_log10_integrated_pl,Top_log10_integrated_pl,Abs_signal_450nm,Abs_signal_650nm,Abs_signal_750nm,Top_minus_Bottom_peak_nm,Top_minus_Bottom_centroid_nm
0,A1,A,1,0.900,0.1,0.0,0.000,1.0000,0.0000,0.0,...,796.0,798.670772,793.513213,5.074690,4.641999,1.137,0.784,0.638,-8.0,-5.157559
1,A2,A,2,0.875,0.1,0.0,0.025,1.0000,0.0000,0.0,...,800.0,799.003212,793.793578,5.127319,4.654614,0.982,0.944,0.818,1.0,-5.209634
2,A3,A,3,0.850,0.1,0.0,0.050,1.0000,0.0000,0.0,...,785.0,786.070694,777.495514,4.930852,4.427226,0.754,0.553,0.471,-4.0,-8.575180
3,A4,A,4,0.800,0.1,0.0,0.100,1.0000,0.0000,0.0,...,777.0,778.954967,771.642733,5.030859,4.551035,0.721,0.490,0.416,-7.0,-7.312234
4,A5,A,5,0.900,0.1,0.0,0.000,0.8335,0.1665,0.0,...,752.0,751.787562,746.566288,5.200939,4.726454,0.987,0.612,0.456,-6.0,-5.221274


In [9]:
# Reusable Challenge 1 composition overview
QUATERNARY_METRICS = {
    "Top_minus_Bottom_peak_nm": "Top - Bottom dominant peak (nm)",
    "Top_minus_Bottom_centroid_nm": "Top - Bottom centroid (nm)",
    "Top_dominant_peak_nm": "Top dominant peak (nm)",
    "Bottom_dominant_peak_nm": "Bottom dominant peak (nm)",
    "Abs_signal_650nm": "Absorbance signal at 650 nm",
}
QUATERNARY_COLORBAR_TITLES = {
    "Top_minus_Bottom_peak_nm": "Δ peak<br>(nm)",
    "Top_minus_Bottom_centroid_nm": "Δ centroid<br>(nm)",
    "Top_dominant_peak_nm": "Top peak<br>(nm)",
    "Bottom_dominant_peak_nm": "Bottom peak<br>(nm)",
    "Abs_signal_650nm": "Absorbance<br>at 650 nm",
}

def quaternary_overview(metric="Top_minus_Bottom_peak_nm"):
    diverging = metric.startswith("Top_minus_Bottom")
    figure = px.scatter(
        quaternary_features,
        x="Rb_design_percent", y="DMA_design_percent", color=metric,
        facet_row="Cs_design_percent", facet_col="Br_design_percent",
        category_orders={
            "Cs_design_percent": sorted(
                quaternary_features["Cs_design_percent"].unique(), reverse=True
            ),
            "Br_design_percent": sorted(
                quaternary_features["Br_design_percent"].unique()
            ),
        },
        custom_data=["well", "FA_fraction", "I_fraction", "Br_fraction"],
        labels={
            "Rb_design_percent": "Rb design level (%)",
            "DMA_design_percent": "DMA design level (%)",
            metric: QUATERNARY_METRICS[metric],
        },
        color_continuous_scale="RdBu_r" if diverging else "Viridis",
        color_continuous_midpoint=0 if diverging else None,
        title=(
            "All 96 compositions"
            f"<br><sup>Color: {QUATERNARY_METRICS[metric]}</sup>"
        ),
        height=760,
    )
    figure.update_traces(
        marker={"size": 16, "line": {"color": "white", "width": 1}},
        hovertemplate=(
            "<b>Well %{customdata[0]}</b><br>Rb %{x:g}%<br>DMA %{y:g}%<br>"
            "FA %{customdata[1]:.3f}<br>I %{customdata[2]:.3f}<br>"
            "Br %{customdata[3]:.3f}<br>Value %{marker.color:.4g}<extra></extra>"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text
            .replace("Cs_design_percent=", "Cs = ")
            .replace("Br_design_percent=", "Br = ")
            + "%"
        )
    )
    figure.update_coloraxes(colorbar={
        "title": {
            "text": QUATERNARY_COLORBAR_TITLES[metric], "side": "right"
        },
        "thickness": 18,
        "len": 0.72,
        "x": 1.02,
    })
    figure.update_xaxes(automargin=True)
    figure.update_yaxes(automargin=True)
    figure.update_layout(
        template="plotly_white", autosize=True,
        margin={"l": 75, "r": 135, "t": 110, "b": 80},
        title={"x": 0.01, "xanchor": "left"},
    )
    return figure


In [10]:
# Reusable Challenge 1 full-spectrum view
def quaternary_spectra(well="H8"):
    sample = quaternary_optical[quaternary_optical["well"].eq(well)]
    composition = quaternary_composition[
        quaternary_composition["well"].eq(well)
    ].iloc[0]
    figure = make_subplots(
        rows=2, cols=1, vertical_spacing=0.18,
        subplot_titles=("Top and Bottom PL shape", "Absorbance channel"),
    )
    for geometry, color in [("Top", "#2563eb"), ("Bottom", "#f97316")]:
        rows = sample[sample["geometry"].eq(geometry)].sort_values("wavelength_nm")
        wavelength = rows["wavelength_nm"].to_numpy(float)
        raw = rows["optical_signal_instrument_units"].to_numpy(float)
        signal = np.clip(raw - np.nanpercentile(raw, 5), 0, None)
        signal = signal / signal.max()
        figure.add_trace(go.Scatter(
            x=wavelength, y=signal, mode="lines", name=f"{geometry} PL",
            line={"color": color, "width": 2.5},
        ), row=1, col=1)

    absorbance = sample[sample["geometry"].eq("Absorbance")].sort_values("wavelength_nm")
    figure.add_trace(go.Scatter(
        x=absorbance["wavelength_nm"],
        y=absorbance["optical_signal_instrument_units"],
        mode="lines", name="Absorbance", line={"color": "#059669", "width": 2.5},
    ), row=2, col=1)
    subtitle = (
        f"FA {composition.FA_fraction:.3f} | Cs {composition.Cs_fraction:.3f} | "
        f"Rb {composition.Rb_fraction:.3f} | DMA {composition.DMA_fraction:.3f} | "
        f"I {composition.I_fraction:.3f} | Br {composition.Br_fraction:.3f}"
    )
    figure.update_xaxes(title_text="Wavelength (nm)", row=2, col=1)
    figure.update_yaxes(title_text="Peak-normalized PL", row=1, col=1)
    figure.update_yaxes(title_text="Instrument units", row=2, col=1)
    figure.update_layout(
        title=f"Well {well}<br><sup>{subtitle}</sup>", height=760,
        template="plotly_white", hovermode="x unified", autosize=True,
        margin={"l": 85, "r": 45, "t": 105, "b": 75},
        title_x=0.01,
    )
    return figure


# Worked Example — Quaternary System

### Question

Where do Top and Bottom PL have different dominant peaks, and what do the complete
spectra look like at one selected composition?

### Baseline solution

The overview uses small multiples for Cs and Br, Rb/DMA position, and color for the
optical difference. The second figure shows Top PL, Bottom PL, and absorbance for well
H8. Hover retains the well and composition details. The figures are intentionally
stacked so neither is squeezed into half of the notebook width.


In [12]:
QUATERNARY_EXAMPLE_WELL = "H8"
quaternary_example_figures = [
    quaternary_overview("Top_minus_Bottom_peak_nm"),
    quaternary_spectra(QUATERNARY_EXAMPLE_WELL),
]
show_figures(quaternary_example_figures)

record = quaternary_features[
    quaternary_features["well"].eq(QUATERNARY_EXAMPLE_WELL)
].iloc[0]
display(Markdown(f'''
### What did we just do?

At **{QUATERNARY_EXAMPLE_WELL}**, the Top-minus-Bottom
dominant-peak difference is **{record.Top_minus_Bottom_peak_nm:+.0f} nm**, while the
centroid difference is **{record.Top_minus_Bottom_centroid_nm:+.2f} nm**. Those two
summaries can even disagree in sign, which is why the full line shapes remain visible.
This is an optical comparison, not a phase assignment.
'''))



### What did we just do?

At **H8**, the Top-minus-Bottom
dominant-peak difference is **+11 nm**, while the
centroid difference is **-2.89 nm**. Those two
summaries can even disagree in sign, which is why the full line shapes remain visible.
This is an optical comparison, not a phase assignment.


# <font color="red">Challenge 1 — Quaternary System</font>

### Your task

Improve the baseline so a new user can connect the four A-site fractions, halide
level, measurement geometry, and full spectra more quickly or more accurately.

### Possible solutions

- add coordinated selection between the overview and spectrum;
- try tetrahedral slices or a different composition encoding;
- make absorbance context more direct;
- compare several wells or geometries without losing the full line shapes;
- build a wavelength-first map or compact spectral glyph view.

### Scientific question

> What trend can a new user discover in 60 seconds, and which spectrum supports it?

### Minimum deliverable

One readable visualization, one evidence-bearing spectrum, one sentence describing the
insight, and one sentence naming a possible misinterpretation.


In [13]:
# ============================================================
# YOUR CODE STARTS HERE — Challenge 1
# ============================================================

CHALLENGE1_METRIC = "Top_minus_Bottom_peak_nm"
CHALLENGE1_WELL = "H8"

# Start from this working baseline, then replace or extend it.
challenge1_figures = [
    quaternary_overview(CHALLENGE1_METRIC),
    quaternary_spectra(CHALLENGE1_WELL),
]

# Ideas: change the encoding, link a selection, add absorbance context,
# compare multiple wells, or replace both figures with one integrated view.

# ============================================================
# YOUR CODE ENDS HERE
# ============================================================
show_figures(challenge1_figures)


<h2><font color="purple">3.</font> Explore the RP-Additive System</h2>

### Problem 2

> **Where do emission bands appear, disappear, or shift with composition and time,
> and do Top and Bottom measurements tell the same story?**

Columns vary the nominal 3AMP/FAPbI3 host mix. Rows vary the nominal MACl design
level. Each well has Top and Bottom PL every 20 minutes from 0 to 380 minutes.

The host fractions form one closed basis. MACl is a separate experimental axis. A
single final-time map hides the trajectory, while 20 separate maps are cumbersome.
The baseline solution pairs a **composition overview at one time** with a
**time × wavelength heatmap** for one selected well.


In [14]:
# Reusable Challenge 2 composition map at one elapsed time
RP_METRICS = {
    "red_fraction_760_820": "Fraction of PL from 760–820 nm",
    "peak_nm": "Dominant emission (nm)",
    "peak_shift_vs_t0": "Peak shift from t0 (nm)",
    "retention_vs_t0": "Integrated PL / t0",
}
RP_COLORBAR_TITLES = {
    "red_fraction_760_820": "760–820 nm<br>PL fraction",
    "peak_nm": "Peak<br>(nm)",
    "peak_shift_vs_t0": "Peak shift<br>(nm)",
    "retention_vs_t0": "PL / t0",
}

def rp_composition_map(
    measure="Bottom PL", time_min=380, metric="red_fraction_760_820",
    selected_well="H7",
):
    current = rp_features[
        rp_features["measure"].eq(measure)
        & np.isclose(rp_features["time_min"], float(time_min))
    ].copy()
    valid = current[current["qc_status"].eq("valid")]
    all_valid = rp_features[
        rp_features["measure"].eq(measure)
        & rp_features["qc_status"].eq("valid")
    ]
    diverging = metric == "peak_shift_vs_t0"
    if diverging:
        bound = float(np.nanmax(np.abs(all_valid[metric])))
        color_range = [-bound, bound]
    else:
        color_range = [float(all_valid[metric].min()), float(all_valid[metric].max())]

    figure = px.scatter(
        valid, x="FAPbI3_percent", y="additive_percent", color=metric,
        custom_data=["well", "spacer_percent", "qc_status"],
        labels={
            "FAPbI3_percent": "FAPbI3 host (%)",
            "additive_percent": "MACl level (%)",
            metric: RP_METRICS[metric],
        },
        color_continuous_scale="RdBu_r" if diverging else "Viridis",
        range_color=color_range,
        title=(
            f"{measure} at {time_min:g} min"
            f"<br><sup>Color: {RP_METRICS[metric]}</sup>"
        ),
        height=650,
    )
    figure.update_traces(
        marker={"size": 16, "line": {"color": "white", "width": 1}},
        hovertemplate=(
            "<b>Well %{customdata[0]}</b><br>FAPbI3 %{x:.1f}%<br>"
            "3AMP host %{customdata[1]:.1f}%<br>MACl level %{y:g}%<br>"
            "Value %{marker.color:.4g}<extra></extra>"
        ),
    )
    invalid = current[~current["qc_status"].eq("valid")]
    if not invalid.empty:
        figure.add_trace(go.Scatter(
            x=invalid["FAPbI3_percent"], y=invalid["additive_percent"],
            mode="markers", name="Low/invalid PL",
            marker={"size": 13, "symbol": "x", "color": "#6b7280"},
            text=invalid["well"], hovertemplate="Well %{text}<extra></extra>",
        ))
    selected = current[current["well"].eq(selected_well)]
    if not selected.empty:
        figure.add_trace(go.Scatter(
            x=selected["FAPbI3_percent"], y=selected["additive_percent"],
            mode="markers", name=f"Selected: {selected_well}",
            marker={"size": 20, "symbol": "circle-open", "color": "black",
                    "line": {"width": 3}},
            hoverinfo="skip",
        ))
    figure.update_coloraxes(colorbar={
        "title": {"text": RP_COLORBAR_TITLES[metric], "side": "right"},
        "thickness": 18,
        "len": 0.72,
        "x": 1.02,
    })
    figure.update_xaxes(range=[-5, 105], automargin=True)
    figure.update_yaxes(range=[-1, 21.5], automargin=True)
    figure.update_layout(
        template="plotly_white", autosize=True,
        margin={"l": 80, "r": 145, "t": 105, "b": 120},
        title={"x": 0.01, "xanchor": "left"},
        legend={
            "orientation": "h", "yanchor": "top", "y": -0.20,
            "xanchor": "left", "x": 0,
        },
    )
    return figure


In [15]:
# Reusable Challenge 2 time × wavelength view for one composition
def rp_time_wavelength(well="H7", measure="Bottom PL"):
    rows = rp_features[
        rp_features["well"].eq(well) & rp_features["measure"].eq(measure)
    ].sort_values("time_min")
    matrix = rp_spectra_by_id.loc[
        rows["spectrum_id"], rp_wavelength_columns
    ].to_numpy(float, copy=True)
    valid = rows["qc_status"].eq("valid").to_numpy()
    matrix[~valid] = np.nan
    maxima = np.ones((len(rows), 1))
    maxima[valid] = np.max(matrix[valid], axis=1, keepdims=True)
    matrix = matrix / maxima

    composition = rp_composition[rp_composition["well"].eq(well)].iloc[0]
    subtitle = (
        f"FAPbI3 host {composition.FAPbI3_percent:.1f}% | "
        f"3AMP host {composition.spacer_percent:.1f}% | "
        f"MACl nominal level {composition.additive_percent:g}%"
    )
    figure = go.Figure(go.Heatmap(
        x=rp_wavelengths, y=rows["time_min"], z=matrix,
        zmin=0, zmax=1, colorscale="Magma",
        colorbar={
            "title": {"text": "PL shape<br>(0–1)", "side": "right"},
            "thickness": 18,
            "len": 0.78,
            "x": 1.02,
        },
        hovertemplate="%{x:.0f} nm<br>%{y:.0f} min<br>%{z:.3f}<extra></extra>",
    ))
    figure.update_layout(
        title=f"Well {well} | {measure}<br><sup>{subtitle}</sup>",
        xaxis_title="Wavelength (nm)", yaxis_title="Elapsed time (min)",
        height=650, template="plotly_white", plot_bgcolor="#d1d5db",
        autosize=True, margin={"l": 85, "r": 125, "t": 105, "b": 75},
        title_x=0.01,
    )
    return figure


# Worked Example — RP-Additive System

### Question

Which compositions show strong long-wavelength PL at 380 minutes, and how did the
spectrum of one selected well evolve to that state?

### Baseline solution

The map uses fixed color limits across time. Low/invalid PL is gray rather than being
silently converted to zero. The second figure retains the complete time–wavelength
history for well H7. The figures are stacked at full width for legibility.


In [16]:
RP_EXAMPLE_WELL = "H7"
RP_EXAMPLE_MEASURE = "Bottom PL"
rp_example_figures = [
    rp_composition_map(
        RP_EXAMPLE_MEASURE, 380, "red_fraction_760_820", RP_EXAMPLE_WELL
    ),
    rp_time_wavelength(RP_EXAMPLE_WELL, RP_EXAMPLE_MEASURE),
]
show_figures(rp_example_figures)

history = rp_features[
    rp_features["well"].eq(RP_EXAMPLE_WELL)
    & rp_features["measure"].eq(RP_EXAMPLE_MEASURE)
].sort_values("time_min")
initial, final = history.iloc[0], history.iloc[-1]
display(Markdown(f'''
### What did we just do?

At **{RP_EXAMPLE_WELL}**, the Bottom-PL dominant maximum
changes from **{initial.peak_nm:.0f} nm** at {initial.time_min:.0f} minutes to
**{final.peak_nm:.0f} nm** at {final.time_min:.0f} minutes. Its 760–820 nm fraction
changes from **{initial.red_fraction_760_820:.3f}** to
**{final.red_fraction_760_820:.3f}**. The heatmap shows whether that scalar change is
a moving band, a competing band, or a change in spectral weight. It is not by itself
evidence of a structural phase transition.
'''))



### What did we just do?

At **H7**, the Bottom-PL dominant maximum
changes from **706 nm** at 0 minutes to
**772 nm** at 380 minutes. Its 760–820 nm fraction
changes from **0.163** to
**0.388**. The heatmap shows whether that scalar change is
a moving band, a competing band, or a change in spectral weight. It is not by itself
evidence of a structural phase transition.


# <font color="red">Challenge 2 — RP-Additive System</font>

### Your task

Improve the baseline so a new user can connect host composition, MACl design level,
time, wavelength, and measurement geometry more quickly or more accurately.

### Possible solutions

- add a time control or animation with a fixed color scale;
- compare Top and Bottom measurements directly;
- encode final − initial change or show a trajectory through time;
- create a wavelength-first composition map;
- link a composition selection to its full spectral history.

### Scientific question

> Where does a spectral feature change, when does it change, and what composition
> context makes that trend understandable?

### Minimum deliverable

One readable visualization, one time-resolved spectral view, one sentence describing
the insight, and one sentence naming a possible misinterpretation.


In [17]:
# ============================================================
# YOUR CODE STARTS HERE — Challenge 2
# ============================================================

CHALLENGE2_MEASURE = "Bottom PL"
CHALLENGE2_TIME_MIN = 380
CHALLENGE2_METRIC = "red_fraction_760_820"
CHALLENGE2_WELL = "H7"

# Start from this working baseline, then replace or extend it.
challenge2_figures = [
    rp_composition_map(
        CHALLENGE2_MEASURE, CHALLENGE2_TIME_MIN,
        CHALLENGE2_METRIC, CHALLENGE2_WELL,
    ),
    rp_time_wavelength(CHALLENGE2_WELL, CHALLENGE2_MEASURE),
]

# Ideas: compare geometries, add a time control, encode two metrics,
# or replace both panels with one more integrated representation.

# ============================================================
# YOUR CODE ENDS HERE
# ============================================================
show_figures(challenge2_figures)


<h2><font color="purple">4.</font> Build, Test, and Explain Your Design</h2>

Use a **60-second usability test**: hand the visualization to someone from another team
and ask them to answer your scientific question without coaching.

Score one point for each:

1. **Dimensional fidelity:** important variables are shown, encoded, or selectable.
2. **Overview + evidence:** a trend can be found and its spectra inspected.
3. **Spectral honesty:** summaries do not masquerade as phase assignments.
4. **Comparable scales + QC:** time/geometry comparisons use defensible scales, and
   low/invalid PL is visibly distinct from measured zero.
5. **One-minute usability:** a new user can answer the stated question.


In [18]:
# @title 5. Team result — edit these three fields
TEAM_NAME = "" # @param {type:"string"}
CHOSEN_CHALLENGE = "Challenge 1 — Quaternary System" # @param ["Challenge 1 — Quaternary System", "Challenge 2 — RP-Additive System"]
DESIGN_CLAIM = "" # @param {type:"string"}

TEAM_FIGURES = (
    challenge1_figures
    if CHOSEN_CHALLENGE.startswith("Challenge 1")
    else challenge2_figures
)
display(Markdown(
    f"### {TEAM_NAME or 'Unnamed team'}\n\n"
    f"**{CHOSEN_CHALLENGE}**\n\n"
    f"{DESIGN_CLAIM or 'Add one sentence explaining what became easier to see.'}"
))
show_figures(TEAM_FIGURES)


### Unnamed team

**Challenge 1 — Quaternary System**

Add one sentence explaining what became easier to see.

In [19]:
# @title 6. Export the final interactive result
safe_team = "".join(
    character if character.isalnum() or character in "-_" else "_"
    for character in (TEAM_NAME.strip() or "unnamed_team")
)
export_dir = DATA_ROOT / "team_exports"
export_dir.mkdir(exist_ok=True)
export_path = export_figure_grid(
    TEAM_FIGURES,
    export_dir / f"{safe_team}_final_visualization.html",
    f"{TEAM_NAME or 'Unnamed team'} — {CHOSEN_CHALLENGE}",
)
print("Wrote", export_path)
display(FileLink(str(export_path), result_html_prefix="Download interactive HTML: "))


Wrote /Users/borisslautin/Documents/GitHub/athena_camm_hackathon/docs/day_19_02102026/team_exports/unnamed_team_final_visualization.html


/Users/borisslautin/Documents/GitHub/athena_camm_hackathon/docs/day_19_02102026/team_exports/unnamed_team_final_visualization.html

## Final Discussion

Be ready to answer:

1. What became easier to see?
2. Which dimensions are directly shown, encoded, selected, or omitted?
3. What could the design cause someone to assume incorrectly?
4. Does the conclusion still hold when the full spectra are inspected?
5. What would you change with another hour?

---

## <font color="green">Takeaway</font>

A useful higher-dimensional visualization does not merely make the plot look compact.
It preserves the scientific variables, exposes comparison choices, distinguishes bad
signal from measured zero, and keeps the underlying spectra inspectable.


## Optional Appendix — Data Provenance and Limits

**Quaternary System:** 96 static wells; one Top PL, Bottom PL, and absorbance read per well.
Nominal composition fractions assume equal-molar stocks. Stock concentrations and
solvents are not included.

**RP-Additive System:** 96 wells × 20 times × two geometries = 3,840 PL spectra. The MACl
percentage is a nominal design label with undocumented denominator. The dispense table
was reconstructed from project plate-layout code and cross-checked against all 96
metadata labels; it was not parsed from the plate-reader workbook.

Each downloaded challenge folder includes `README_DATA.md` and `SHA256SUMS.txt` for a
fuller field description and file-integrity record.


In [20]:
# ============================================================
# Challenge 1 — look at the composition space only (no spectra)
# ============================================================
# Uses: quaternary_composition (from "3. Load the prepared tables")
#       show_figures (from the "display/export helpers" cell)

comp = quaternary_composition.copy()
A_SITE = ["FA_fraction", "Cs_fraction", "Rb_fraction", "DMA_fraction"]


# ---------- Figure 1: A-site space (FA + Cs + Rb + DMA = 1) ----------

# Each A-site cation sits on one corner of a regular tetrahedron.
CORNERS = {
    "FA":  [0.0, 0.0, 0.0],
    "Cs":  [1.0, 0.0, 0.0],
    "Rb":  [0.5, np.sqrt(3) / 2, 0.0],
    "DMA": [0.5, np.sqrt(3) / 6, np.sqrt(6) / 3],
}
corner_xyz = np.array(list(CORNERS.values()))

# The three Br levels use the same A-site mix, so we keep one point per mix
# and list the three wells that share it.
a_site = (
    comp.groupby(A_SITE)["well"]
    .agg(lambda wells: ", ".join(wells))
    .reset_index(name="wells")
)

# A composition is a weighted average of the corners (weights = fractions).
xyz = a_site[A_SITE].to_numpy() @ corner_xyz

# Text shown when you hover over a point
a_site_hover = [
    f"<b>Wells {row.wells}</b><br>"
    f"FA {row.FA_fraction:.3f} | Cs {row.Cs_fraction:.3f}<br>"
    f"Rb {row.Rb_fraction:.3f} | DMA {row.DMA_fraction:.3f}<br>"
    "(one well per Br level)"
    for row in a_site.itertuples()
]

# Six tetrahedron edges. A row of NaN breaks the line between edges.
edge_xyz = []
for i in range(4):
    for j in range(i + 1, 4):
        edge_xyz += [corner_xyz[i], corner_xyz[j], [np.nan] * 3]
edge_xyz = np.array(edge_xyz)

# Shared point style: color = FA fraction (one shared color bar)
point_style = {
    "size": 6,
    "color": a_site["FA_fraction"],
    "coloraxis": "coloraxis",
    "line": {"color": "white", "width": 0.5},
}

fig_a_site = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=(
        "Full A-site tetrahedron (where the samples sit)",
        "Zoom: Cs, Rb, DMA in % (FA = the rest)",
    ),
    horizontal_spacing=0.02,
)

# Left: whole tetrahedron with its corners labelled
fig_a_site.add_trace(go.Scatter3d(
    x=edge_xyz[:, 0], y=edge_xyz[:, 1], z=edge_xyz[:, 2],
    mode="lines", line={"color": "gray", "width": 3},
    hoverinfo="skip", showlegend=False,
), row=1, col=1)
fig_a_site.add_trace(go.Scatter3d(
    x=corner_xyz[:, 0], y=corner_xyz[:, 1], z=corner_xyz[:, 2],
    mode="text", text=[f"<b>{name}</b>" for name in CORNERS],
    textfont={"size": 15}, hoverinfo="skip", showlegend=False,
), row=1, col=1)
fig_a_site.add_trace(go.Scatter3d(
    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
    mode="markers", marker=point_style,
    hovertext=a_site_hover, hoverinfo="text", showlegend=False,
), row=1, col=1)

# Right: the same 32 points, plotted with the three free fractions as axes
fig_a_site.add_trace(go.Scatter3d(
    x=a_site["Cs_fraction"] * 100,
    y=a_site["Rb_fraction"] * 100,
    z=a_site["DMA_fraction"] * 100,
    mode="markers", marker={**point_style, "size": 7},
    hovertext=a_site_hover, hoverinfo="text", showlegend=False,
), row=1, col=2)

hidden_axis = {"visible": False}
fig_a_site.update_layout(
    title=(
        "A-site composition space: 32 mixes (each made with 3 Br levels = 96 wells)"
        "<br><sup>All samples sit in a small region near the FA corner, "
        "because Rb and DMA never go above 10%</sup>"
    ),
    scene={
        "xaxis": hidden_axis, "yaxis": hidden_axis, "zaxis": hidden_axis,
        "aspectmode": "data",
        "camera": {"eye": {"x": -1.1, "y": -1.7, "z": 1.0}},
    },
    scene2={
        "xaxis_title": "Cs (%)", "yaxis_title": "Rb (%)", "zaxis_title": "DMA (%)",
        "aspectmode": "cube",
    },
    coloraxis={
        "colorscale": "Viridis",
        "colorbar": {"title": {"text": "FA<br>fraction"}, "thickness": 16},
    },
    height=620, template="plotly_white",
    margin={"l": 10, "r": 10, "t": 110, "b": 10},
)


# ---------- Figure 2: all 96 wells on one nested grid ----------
# Rows    = Cs level, then Rb level inside it
# Columns = Br level, then DMA level inside it
# This puts every composition knob on screen at once (no overlap).

COLOR_LABELS = {
    "FA_fraction": "FA fraction",
    "Cs_design_percent": "Cs (%)",
    "Rb_design_percent": "Rb (%)",
    "DMA_design_percent": "DMA (%)",
    "Br_design_percent": "Br (%)",
}

def composition_grid(color_by="FA_fraction"):
    row_keys = ["Cs_design_percent", "Rb_design_percent"]
    col_keys = ["Br_design_percent", "DMA_design_percent"]

    # One 8 x 12 table for the color and one for the well names
    color_table = comp.pivot(index=row_keys, columns=col_keys, values=color_by)
    well_table = comp.pivot(index=row_keys, columns=col_keys, values="well")
    well_table = well_table.loc[color_table.index, color_table.columns]

    # Full composition text for hovering, in the same 8 x 12 layout
    label_table = comp.pivot(
        index=row_keys, columns=col_keys, values="composition_label"
    ).loc[color_table.index, color_table.columns]
    hover_text = (well_table + "<br>" + label_table.replace(r"\|\|", "<br>", regex=True))

    figure = go.Figure(go.Heatmap(
        z=color_table.to_numpy(float),
        x=list(range(color_table.shape[1])),
        y=list(range(color_table.shape[0])),
        text=well_table.to_numpy(),
        texttemplate="%{text}",
        textfont={"size": 11},
        hovertext=hover_text.to_numpy(),
        hoverinfo="text",
        colorscale="Viridis", xgap=2, ygap=2,
        colorbar={"title": {"text": COLOR_LABELS.get(color_by, color_by)}},
    ))

    # Small tick labels: the inner knob of each row / column
    figure.update_xaxes(
        tickvals=list(range(color_table.shape[1])),
        ticktext=[f"DMA {dma:g}" for _, dma in color_table.columns],
        side="bottom", tickangle=0,
    )
    figure.update_yaxes(
        tickvals=list(range(color_table.shape[0])),
        ticktext=[f"Rb {rb:g}" for _, rb in color_table.index],
        autorange="reversed",
    )

    # Big group labels + divider lines: the outer knob (Br on top, Cs on the left)
    br_levels = color_table.columns.get_level_values(0)
    cs_levels = color_table.index.get_level_values(0)
    for level in br_levels.unique():
        positions = np.where(br_levels == level)[0]
        figure.add_annotation(
            x=positions.mean(), y=1.06, yref="paper", showarrow=False,
            text=f"<b>Br {level:.3g}%</b>", font={"size": 14},
        )
        if positions[0] > 0:
            figure.add_vline(x=positions[0] - 0.5, line={"color": "black", "width": 3})
    for level in cs_levels.unique():
        positions = np.where(cs_levels == level)[0]
        figure.add_annotation(
            x=-0.11, xref="paper", y=positions.mean(), showarrow=False,
            text=f"<b>Cs {level:g}%</b>", font={"size": 14}, textangle=-90,
        )
        if positions[0] > 0:
            figure.add_hline(y=positions[0] - 0.5, line={"color": "black", "width": 3})

    figure.update_layout(
        title=(
            f"All 96 wells by composition — color: {COLOR_LABELS.get(color_by, color_by)}"
            "<br><sup>Rows: Cs then Rb (%). Columns: Br then DMA (%). "
            "Hover a cell for the full composition.</sup>"
        ),
        height=560, template="plotly_white", plot_bgcolor="white",
        margin={"l": 110, "r": 40, "t": 120, "b": 50},
    )
    return figure


# Change color_by to "Cs_design_percent", "Rb_design_percent",
# "DMA_design_percent" or "Br_design_percent" to color by another knob.
show_figures([fig_a_site, composition_grid(color_by="FA_fraction")])

In [25]:
# ============================================================
# Challenge 1 — small spectra on the composition grid (simple version)
# ============================================================
# The plate layout already IS the composition grid:
#   plate rows A–H     = Cs (10, 30%), then Rb (0, 2.5, 5, 10%)
#   plate columns 1–12 = Br (0, 16.7, 33.3%), then DMA (0, 2.5, 5, 10%)
# So we draw every well at its own plate position.

WL_MIN, WL_MAX = 640, 850          # wavelength window shown in every cell (nm)
BACKGROUND = "residual_meV"        # cell color; other choice: "top_bottom_ratio"

# Split the long table once: one small table per (well, geometry)
by_well = {
    key: table.sort_values("wavelength_nm")
    for key, table in quaternary_optical.groupby(["well", "geometry"])
}

def get_curve(well, geometry):
    """Wavelength and signal inside the window. PL gets its flat background removed."""
    table = by_well[(well, geometry)]
    wl = table["wavelength_nm"].to_numpy(float)
    signal = table["optical_signal_instrument_units"].to_numpy(float)
    if geometry != "Absorbance":
        signal = np.clip(signal - np.percentile(signal, 5), 0, None)
    keep = (wl >= WL_MIN) & (wl <= WL_MAX)
    return wl[keep], signal[keep]


# ---------- Step 1: one peak energy per well ----------
cells = quaternary_composition.copy()
peak_eV, top_bottom_ratio = [], []
for well in cells["well"]:
    wl, bottom = get_curve(well, "Bottom")      # Bottom PL: much less noise than Top
    _, top = get_curve(well, "Top")
    near = np.abs(wl - wl[bottom.argmax()]) <= 50                    # ±50 nm around the peak
    peak_nm = np.sum(wl[near] * bottom[near]) / np.sum(bottom[near])  # weighted mean wavelength
    peak_eV.append(1239.84 / peak_nm)                                 # nm -> eV
    top_bottom_ratio.append(top[near].sum() / bottom[near].sum())
cells["peak_eV"] = peak_eV
cells["top_bottom_ratio"] = top_bottom_ratio


# ---------- Step 2: smooth trend and residual ----------
# peak energy ≈ E0 + a·Cs + b·Rb + c·DMA + d·Br + bowing·Br(1−Br) + e·Cs·Br
cs, rb, dma, br = cells["Cs_fraction"], cells["Rb_fraction"], cells["DMA_fraction"], cells["Br_fraction"]
X = np.column_stack([np.ones(len(cells)), cs, rb, dma, br, br * (1 - br), cs * br])
coef = np.linalg.lstsq(X, cells["peak_eV"], rcond=None)[0]
cells["residual_meV"] = 1000 * (cells["peak_eV"] - X @ coef)   # + = bluer than the trend
print(f"Typical distance from the trend: {cells['residual_meV'].std():.1f} meV")


# ---------- Step 3: the figure ----------
cells["plate_row"] = cells["row"].map("ABCDEFGH".index)   # A -> 0, ..., H -> 7
cells["plate_col"] = cells["col"] - 1                       # 1 -> 0, ..., 12 -> 11
cells["hover"] = (
    "<b>" + cells["well"] + "</b><br>" + cells["composition_label"]
    + "<br>residual: " + cells["residual_meV"].round(1).astype(str) + " meV"
    + "<br>Top/Bottom: " + cells["top_bottom_ratio"].round(2).astype(str)
)

# Background: one colored square per well
is_residual = BACKGROUND == "residual_meV"
fig = go.Figure(go.Heatmap(
    z=cells.pivot(index="plate_row", columns="plate_col", values=BACKGROUND).to_numpy(),
    x=np.arange(12) + 0.5, y=np.arange(8) + 0.5,
    hovertext=cells.pivot(index="plate_row", columns="plate_col", values="hover").to_numpy(),
    hoverinfo="text",
    colorscale="RdBu_r" if is_residual else "Blues",
    zmid=0 if is_residual else None,
    opacity=0.6, xgap=3, ygap=3,
    colorbar={"title": {"text": "residual (meV)<br>+ = bluer" if is_residual else BACKGROUND}},
))

# Small spectra: squeeze each curve into its own cell
for c in cells.itertuples():
    wl, bottom = get_curve(c.well, "Bottom")
    _, top = get_curve(c.well, "Top")
    wl_abs, absorb = get_curve(c.well, "Absorbance")
    near = np.abs(wl - wl[bottom.argmax()]) <= 50

    def to_cell_x(w):
        return c.plate_col + 0.05 + 0.9 * (w - WL_MIN) / (WL_MAX - WL_MIN)

    def to_cell_y(v):                      # v in 0..1; row A is at the top
        return c.plate_row + 0.9 - 0.8 * v

    # (name, x, y scaled 0..1, color, line width, line style)
    curves = [
        ("Bottom PL", to_cell_x(wl), bottom / bottom.max(), "darkgray", 3.5, "solid"),
        ("Top PL", to_cell_x(wl), np.clip(top / top[near].max(), 0, 1.15), "black", 1, "solid"),
        ("Absorbance", to_cell_x(wl_abs), (absorb - absorb.min()) / (absorb.max() - absorb.min()),
         "dimgray", 1, "dot"),
    ]
    for name, x, v, color, width, dash in curves:
        fig.add_trace(go.Scatter(
            x=x, y=to_cell_y(v), mode="lines", name=name,
            line={"color": color, "width": width, "dash": dash},
            legendgroup=name, showlegend=(c.well == "A1"), hoverinfo="skip",
        ))

# Well names in the bottom-left corner of each cell (empty area)
fig.add_trace(go.Scatter(
    x=cells["plate_col"] + 0.07, y=cells["plate_row"] + 0.8, text=cells["well"],
    mode="text", textposition="middle right", textfont={"size": 9},
    showlegend=False, hoverinfo="skip",
))

# Axis labels from the composition of the first row / first column
first_row = cells[cells["plate_row"] == 0].sort_values("plate_col")
first_col = cells[cells["plate_col"] == 0].sort_values("plate_row")
fig.update_xaxes(
    range=[0, 12], tickvals=np.arange(12) + 0.5, showgrid=False, zeroline=False,
    ticktext=[f"Br {b:.3g}<br>DMA {d:g}" for b, d in
              zip(first_row["Br_design_percent"], first_row["DMA_design_percent"])],
)
fig.update_yaxes(
    range=[8, 0], tickvals=np.arange(8) + 0.5, showgrid=False, zeroline=False,
    ticktext=[f"Cs {c:g} · Rb {r:g}" for c, r in
              zip(first_col["Cs_design_percent"], first_col["Rb_design_percent"])],
)

# Thick lines between the big composition blocks (Br blocks, Cs halves)
for x in [4, 8]:
    fig.add_vline(x=x, line={"color": "black", "width": 2})
fig.add_hline(y=4, line={"color": "black", "width": 2})

fig.update_layout(
    title=f"PL and absorbance on the composition grid — background: {BACKGROUND}"
          f"<br><sup>Every cell shows {WL_MIN}–{WL_MAX} nm. Hover a cell for numbers.</sup>",
    height=600, template="plotly_white",
    legend={"orientation": "h", "y": -0.1},
)
show_figures([fig])

# Wells farthest from the trend
display(cells.sort_values("residual_meV", key=abs, ascending=False)
        [["well", "residual_meV", "top_bottom_ratio", "composition_label"]].head(6).round(2))

Typical distance from the trend: 6.2 meV


,well,residual_meV,top_bottom_ratio,composition_label
91,H8,-16.29,0.44,FA 0.500 | Cs 0.300 | Rb 0.100 | DMA 0.100 || ...
23,B12,15.03,0.28,FA 0.775 | Cs 0.100 | Rb 0.025 | DMA 0.100 || ...
93,H10,13.58,0.30,FA 0.575 | Cs 0.300 | Rb 0.100 | DMA 0.025 || ...
94,H11,-12.76,0.42,FA 0.550 | Cs 0.300 | Rb 0.100 | DMA 0.050 || ...
77,G6,12.31,0.34,FA 0.625 | Cs 0.300 | Rb 0.050 | DMA 0.025 || ...
68,F9,12.02,0.27,FA 0.675 | Cs 0.300 | Rb 0.025 | DMA 0.000 || ...


In [ ]:
# ============================================================
# Challenge 1 — all spectra together: what changes, and what is unique?
# ============================================================
# Idea:
#   1. Peak-normalize every Bottom PL spectrum (Bottom has much less noise than Top).
#   2. Br sets the color, so subtract the average spectrum of the same Br level.
#      What is left = changes caused by Cs, Rb, DMA + anything unusual.
#   3. Inside each Br level, fit:  difference(λ) = a(λ)·Cs + b(λ)·Rb + c(λ)·DMA
#      a(λ), b(λ), c(λ) show what each knob does to the spectrum (attribution).
#   4. Whatever the knobs cannot explain = "unique" features. We list the biggest ones.
#
# Uses: quaternary_optical, quaternary_composition, show_figures

WL_MIN, WL_MAX = 640, 840     # stop at 840 nm: the last few nm have noise spikes
KNOBS = ["Cs_fraction", "Rb_fraction", "DMA_fraction"]
KNOB_COLORS = {"Cs_fraction": "#2563eb", "Rb_fraction": "#16a34a", "DMA_fraction": "#dc2626"}


# ---------- Step 1: one table of normalized spectra (rows = wells, columns = wavelengths) ----------
bottom = quaternary_optical[quaternary_optical["geometry"].eq("Bottom")].pivot_table(
    index="well", columns="wavelength_nm", values="optical_signal_instrument_units")
bottom = bottom.sub(bottom.quantile(0.05, axis=1), axis=0).clip(lower=0)  # remove flat background
bottom = bottom.loc[:, WL_MIN:WL_MAX]
spectra = bottom.div(bottom.max(axis=1), axis=0)                            # peak = 1
wl = spectra.columns.to_numpy(float)

# Order wells by composition: Br, then Cs, Rb, DMA
comp = quaternary_composition.sort_values(
    ["Br_design_percent", "Cs_design_percent", "Rb_design_percent", "DMA_design_percent"]
).set_index("well")
spectra = spectra.loc[comp.index]


# ---------- Step 2: difference from the average spectrum of the same Br level ----------
br_level = comp["Br_design_percent"]
difference = spectra - spectra.groupby(br_level).transform("mean")


# ---------- Step 3: what does each knob do? (one small fit per Br level) ----------
effects = {}                                   # (Br level, knob) -> change over the full knob range
unexplained = pd.DataFrame(index=difference.index, columns=wl, dtype=float)
for level, wells in comp.groupby("Br_design_percent").groups.items():
    knobs = comp.loc[wells, KNOBS]
    X = np.column_stack([np.ones(len(wells)), knobs - knobs.mean()])
    coef = np.linalg.lstsq(X, difference.loc[wells].to_numpy(), rcond=None)[0]
    for i, knob in enumerate(KNOBS):
        knob_range = knobs[knob].max() - knobs[knob].min()     # e.g. Rb: 0 -> 0.10
        effects[(level, knob)] = coef[i + 1] * knob_range
    unexplained.loc[wells] = difference.loc[wells].to_numpy() - X @ coef


# ---------- Step 4: the most unique wells ----------
unique = pd.DataFrame({
    "size": unexplained.abs().max(axis=1),                    # biggest leftover (0..1 of peak)
    "at_nm": unexplained.abs().idxmax(axis=1),                # where it happens
    "composition": comp["composition_label"],
}).sort_values("size", ascending=False)
TOP_UNIQUE = unique.head(6)
display(TOP_UNIQUE.round(3))


# ---------- Figure 1: all difference spectra in one heatmap ----------
row_labels = [f"{w}  Cs{c:g} Rb{r:g} DMA{d:g}" for w, c, r, d in zip(
    comp.index, comp["Cs_design_percent"], comp["Rb_design_percent"], comp["DMA_design_percent"])]
bound = np.abs(difference.to_numpy()).max()
fig_diff = go.Figure(go.Heatmap(
    z=difference.to_numpy(), x=wl, y=row_labels,
    colorscale="RdBu_r", zmin=-bound, zmax=bound,
    colorbar={"title": {"text": "PL − average of<br>same Br level"}},
    hovertemplate="%{y}<br>%{x:.0f} nm<br>difference %{z:+.2f}<extra></extra>",
))
# Lines between Br levels + Br label on the right
for level in br_level.unique():
    rows = np.where(br_level.to_numpy() == level)[0]
    fig_diff.add_hline(y=rows[0] - 0.5, line={"color": "black", "width": 2})
    fig_diff.add_annotation(x=1.0, xref="paper", xanchor="left", y=row_labels[rows[len(rows) // 2]],
                            text=f"<b>Br {level:.3g}%</b>", showarrow=False, textangle=90)
# Circle the most unique features
fig_diff.add_trace(go.Scatter(
    x=TOP_UNIQUE["at_nm"], y=[row_labels[list(comp.index).index(w)] for w in TOP_UNIQUE.index],
    mode="markers", marker={"symbol": "circle-open", "size": 16, "color": "black", "line": {"width": 2}},
    name="most unique", hoverinfo="skip",
))
fig_diff.update_yaxes(autorange="reversed", tickfont={"size": 8})
fig_diff.update_layout(
    title="Every Bottom PL spectrum minus the average of its Br level"
          "<br><sup>Red = more light than average, blue = less. "
          "A red/blue pair side by side = a peak shift. Circles = most unique features.</sup>",
    xaxis_title="Wavelength (nm)", height=1100, template="plotly_white",
    legend={"orientation": "h", "y": -0.04},
)


# ---------- Figure 2: what each knob does to the spectrum ----------
levels = sorted(br_level.unique())
fig_knobs = make_subplots(rows=1, cols=len(levels), shared_yaxes=True,
                          subplot_titles=[f"Br {lv:.3g}%" for lv in levels])
for col, level in enumerate(levels, start=1):
    # average spectrum of this Br level, as a gray guide (scaled down)
    mean_spectrum = spectra[br_level == level].mean()
    fig_knobs.add_trace(go.Scatter(
        x=wl, y=0.3 * mean_spectrum, fill="tozeroy", mode="none",
        fillcolor="rgba(0,0,0,0.1)", name="average spectrum (scaled)",
        showlegend=(col == 1), hoverinfo="skip"), row=1, col=col)
    for knob in KNOBS:
        knob_range = comp[knob].max() - comp[knob].min()
        fig_knobs.add_trace(go.Scatter(
            x=wl, y=effects[(level, knob)], mode="lines",
            line={"color": KNOB_COLORS[knob], "width": 2.5},
            name=f"{knob.split('_')[0]} +{100 * knob_range:g}%", showlegend=(col == 1),
            hovertemplate=f"{knob.split('_')[0]}: " + "%{x:.0f} nm, %{y:+.2f}<extra></extra>",
        ), row=1, col=col)
    fig_knobs.add_hline(y=0, line={"color": "gray", "width": 1}, row=1, col=col)
fig_knobs.update_xaxes(title_text="Wavelength (nm)")
fig_knobs.update_yaxes(title_text="Change in normalized PL", col=1)
fig_knobs.update_layout(
    title="What each knob does to the spectrum (over its full design range)"
          "<br><sup>A −/+ pair = the peak moves (− on the side it leaves). "
          "One bump = a band grows. Gray = average spectrum, for orientation.</sup>",
    height=480, template="plotly_white", legend={"orientation": "h", "y": -0.2},
)


# ---------- Figure 3: the unique wells, measured vs. what the knobs predict ----------
fig_unique = make_subplots(rows=2, cols=3, subplot_titles=[
    f"{w}: {comp.loc[w, 'composition_label'].replace('||', '<br>')}" for w in TOP_UNIQUE.index])
for i, well in enumerate(TOP_UNIQUE.index):
    row, col = i // 3 + 1, i % 3 + 1
    predicted = spectra.loc[well] - unexplained.loc[well]          # average + knob effects
    fig_unique.add_trace(go.Scatter(x=wl, y=predicted, mode="lines", name="predicted from composition",
                                    line={"color": "#dc2626", "dash": "dash", "width": 2},
                                    showlegend=(i == 0)), row=row, col=col)
    fig_unique.add_trace(go.Scatter(x=wl, y=spectra.loc[well], mode="lines", name="measured",
                                    line={"color": "black", "width": 1.5},
                                    showlegend=(i == 0)), row=row, col=col)
    fig_unique.add_vline(x=TOP_UNIQUE.loc[well, "at_nm"], line={"color": "gray", "dash": "dot"},
                         row=row, col=col)
fig_unique.update_annotations(font_size=10)
fig_unique.update_layout(
    title="Most unique wells: measured Bottom PL vs. what Br + Cs + Rb + DMA predict",
    height=650, template="plotly_white", legend={"orientation": "h", "y": -0.08},
    margin={"t": 120},
)

show_figures([fig_diff, fig_knobs, fig_unique])

,size,at_nm,composition
well,,,
H8,0.282,761.0,FA 0.500 | Cs 0.300 | Rb 0.100 | DMA 0.100 || ...
E4,0.263,781.0,FA 0.600 | Cs 0.300 | Rb 0.000 | DMA 0.100 || ...
H10,0.234,686.0,FA 0.575 | Cs 0.300 | Rb 0.100 | DMA 0.025 || ...
H4,0.222,784.0,FA 0.500 | Cs 0.300 | Rb 0.100 | DMA 0.100 || ...
D1,0.217,785.0,FA 0.800 | Cs 0.100 | Rb 0.100 | DMA 0.000 || ...
F9,0.210,725.0,FA 0.675 | Cs 0.300 | Rb 0.025 | DMA 0.000 || ...


In [28]:
# ============================================================
# Challenge 1 — 96 spectral "flowers" on the composition grid
# ============================================================
# Each well = one flower. Each petal = one slice of the spectrum (a 25 meV band).
#   Petals go from the BLUE side (left) over the PEAK (top) to the RED side (right);
#   the gap at the bottom is the stem.
#   Petal length = Bottom PL in that band minus the average of the same Br level
#     red petal out  = more light than average
#     blue petal in  = less light than average
#   So: flower leans left = bluer peak, leans right = redder, both ends grow = broader.
# Center dot color = Top / Bottom brightness.  Thick ring = most unusual wells.
#
# Uses: quaternary_optical, quaternary_composition, show_figures

N_PETALS, BAND_meV = 8, 25            # 8 bands of 25 meV = peak ± 100 meV
KNOBS = ["Cs_fraction", "Rb_fraction", "DMA_fraction"]


# ---------- Step 1: normalized Bottom PL spectra (rows = wells) ----------
def pl_table(geometry):
    table = quaternary_optical[quaternary_optical["geometry"].eq(geometry)].pivot_table(
        index="well", columns="wavelength_nm", values="optical_signal_instrument_units")
    return table.sub(table.quantile(0.05, axis=1), axis=0).clip(lower=0)   # remove flat background

bottom, top = pl_table("Bottom"), pl_table("Top")
spectra = bottom.div(bottom.max(axis=1), axis=0)            # peak = 1
energy_meV = 1e6 * 1.23984 / spectra.columns.to_numpy(float)  # photon energy of each column

comp = quaternary_composition.set_index("well").loc[spectra.index]
br_level = comp["Br_design_percent"]
average = spectra.groupby(br_level).transform("mean")         # average spectrum of the same Br level
difference = spectra - average


# ---------- Step 2: petal values = average difference inside each energy band ----------
petals = pd.DataFrame(index=spectra.index, columns=range(N_PETALS), dtype=float)
for level in br_level.unique():
    wells = br_level.index[br_level == level]
    peak_meV = energy_meV[average.loc[wells[0]].to_numpy().argmax()]   # peak of the Br average
    offset = energy_meV - peak_meV                                    # + = bluer than the peak
    for p in range(N_PETALS):                                         # petal 0 = bluest band
        high = N_PETALS / 2 * BAND_meV - p * BAND_meV
        in_band = (offset <= high) & (offset > high - BAND_meV)
        petals.loc[wells, p] = difference.loc[wells, in_band].mean(axis=1)


# ---------- Step 3: which knob changes which petal? (small fit per Br level) ----------
key_flowers = {}                                   # (Br level, knob) -> petal change over full range
leftover = pd.DataFrame(index=petals.index, columns=petals.columns, dtype=float)
for level in br_level.unique():
    wells = br_level.index[br_level == level]
    knobs = comp.loc[wells, KNOBS]
    X = np.column_stack([np.ones(len(wells)), knobs - knobs.mean()])
    coef = np.linalg.lstsq(X, petals.loc[wells].to_numpy(), rcond=None)[0]
    for i, knob in enumerate(KNOBS):
        key_flowers[(level, knob)] = coef[i + 1] * (knobs[knob].max() - knobs[knob].min())
    leftover.loc[wells] = petals.loc[wells].to_numpy() - X @ coef

comp["unusual"] = leftover.abs().max(axis=1)                  # biggest petal the knobs cannot explain
comp["top_bottom"] = (top.loc[:, 640:850].sum(axis=1) / bottom.loc[:, 640:850].sum(axis=1))
UNUSUAL = comp["unusual"].nlargest(6).index
display(comp.loc[UNUSUAL, ["unusual", "top_bottom", "composition_label"]].round(3))


# ---------- Drawing helper: add many flowers to a figure ----------
BASE_R, MAX_LEN = 0.2, 0.2                       # base circle radius, longest petal (cell = 1)
SCALE = MAX_LEN / np.abs(petals.to_numpy()).max()
ANGLES = np.radians(225 - (np.arange(N_PETALS) + 0.5) * 270 / N_PETALS)   # left -> top -> right
HALF_WIDTH = np.radians(270 / N_PETALS) * 0.42

def add_flowers(fig, centers_x, centers_y, petal_rows):
    shapes = {"out": ([], []), "in": ([], [])}
    ring_x, ring_y, stem_x, stem_y = [], [], [], []
    circle = np.linspace(0, 2 * np.pi, 40)
    for cx, cy, values in zip(centers_x, centers_y, petal_rows):
        ring_x += list(cx + BASE_R * np.cos(circle)) + [None]
        ring_y += list(cy - BASE_R * np.sin(circle)) + [None]
        stem_x += [cx, cx, None]
        stem_y += [cy + BASE_R, cy + 0.47, None]
        for angle, value in zip(ANGLES, values):
            length = float(np.clip(value * SCALE, -BASE_R, MAX_LEN))
            r_inner, r_outer = sorted([BASE_R, BASE_R + length])
            arc = np.linspace(angle - HALF_WIDTH, angle + HALF_WIDTH, 6)
            radius = np.r_[np.full(6, r_inner), np.full(6, r_outer)]
            theta = np.r_[arc, arc[::-1]]
            xs, ys = shapes["out" if length > 0 else "in"]
            xs += list(cx + radius * np.cos(theta)) + [None]
            ys += list(cy - radius * np.sin(theta)) + [None]   # minus: y axis points down
    fig.add_trace(go.Scatter(x=stem_x, y=stem_y, mode="lines", line={"color": "#9ca3af", "width": 1},
                             hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=ring_x, y=ring_y, mode="lines", line={"color": "#9ca3af", "width": 1},
                             hoverinfo="skip", showlegend=False))
    for kind, color, label in [("out", "#dc2626", "more light than Br average"),
                               ("in", "#2563eb", "less light than Br average")]:
        xs, ys = shapes[kind]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", fill="toself", fillcolor=color,
                                 line={"width": 0}, name=label, hoverinfo="skip"))


# ---------- Figure 1: the 96 flowers on the plate (= composition) grid ----------
comp["plate_row"] = comp["row"].map("ABCDEFGH".index)
comp["plate_col"] = comp["col"] - 1
cx, cy = comp["plate_col"] + 0.5, comp["plate_row"] + 0.45

fig_flowers = go.Figure()
add_flowers(fig_flowers, cx, cy, petals.to_numpy())

# Center dots: color = Top/Bottom brightness; hover shows the numbers
fig_flowers.add_trace(go.Scatter(
    x=cx, y=cy, mode="markers", name="Top/Bottom brightness",
    marker={"size": 9, "color": comp["top_bottom"], "colorscale": "Greys", "cmin": 0.2, "cmax": 0.6,
            "line": {"color": "black", "width": 0.5},
            "colorbar": {"title": {"text": "Top / Bottom<br>brightness"}, "len": 0.5}},
    text=[f"<b>{w}</b><br>{c.composition_label}<br>Top/Bottom {c.top_bottom:.2f}"
          f"<br>unusual: {c.unusual:.2f}" for w, c in comp.iterrows()],
    hoverinfo="text", showlegend=False,
))
# Thick rings around the most unusual wells (drawn in grid units, so they scale with the figure)
for well in UNUSUAL:
    fig_flowers.add_shape(type="circle", x0=cx[well] - 0.46, x1=cx[well] + 0.46,
                          y0=cy[well] - 0.44, y1=cy[well] + 0.48, line={"color": "black", "width": 2.5})
# Well names
fig_flowers.add_trace(go.Scatter(
    x=comp["plate_col"] + 0.04, y=comp["plate_row"] + 0.06, text=comp.index, mode="text",
    textposition="bottom right", textfont={"size": 9, "color": "#374151"},
    hoverinfo="skip", showlegend=False))

first_row = comp[comp["plate_row"] == 0].sort_values("plate_col")
first_col = comp[comp["plate_col"] == 0].sort_values("plate_row")
fig_flowers.update_xaxes(range=[0, 12], tickvals=np.arange(12) + 0.5, showgrid=False, zeroline=False,
                         ticktext=[f"Br {b:.3g}<br>DMA {d:g}" for b, d in
                                   zip(first_row["Br_design_percent"], first_row["DMA_design_percent"])])
fig_flowers.update_yaxes(range=[8, 0], tickvals=np.arange(8) + 0.5, showgrid=False, zeroline=False,
                         scaleanchor="x", scaleratio=1,          # keep the flowers round
                         ticktext=[f"Cs {c:g} · Rb {r:g}" for c, r in
                                   zip(first_col["Cs_design_percent"], first_col["Rb_design_percent"])])
fig_flowers.update_xaxes(constrain="domain")    # shrink the plot area instead of adding empty space
fig_flowers.update_yaxes(constrain="domain")
for x in [4, 8]:
    fig_flowers.add_vline(x=x, line={"color": "black", "width": 1.5})
fig_flowers.add_hline(y=4, line={"color": "black", "width": 1.5})
fig_flowers.update_layout(
    title="Spectral flowers: how each well's Bottom PL differs from the average of its Br level"
          "<br><sup>Petals: blue side (left) → peak (top) → red side (right), 25 meV each. "
          "Red out = more light, blue in = less. Black ring = most unusual well.</sup>",
    height=720, template="plotly_white", plot_bgcolor="white",
    legend={"orientation": "h", "y": -0.08},
)


# ---------- Figure 2: the key — what each knob does to the flower ----------
levels = sorted(br_level.unique())
fig_key = go.Figure()
key_x, key_y, key_values = [], [], []
for row, level in enumerate(levels):
    for col, knob in enumerate(KNOBS):
        key_x.append(col + 0.5)
        key_y.append(row + 0.45)
        key_values.append(key_flowers[(level, knob)])
add_flowers(fig_key, key_x, key_y, key_values)
fig_key.update_xaxes(range=[0, 3], tickvals=[0.5, 1.5, 2.5], showgrid=False, zeroline=False,
                     ticktext=[f"{k.split('_')[0]} +{100 * (comp[k].max() - comp[k].min()):g}%"
                               for k in KNOBS])
fig_key.update_yaxes(range=[3, 0], tickvals=[0.5, 1.5, 2.5], showgrid=False, zeroline=False,
                     scaleanchor="x", scaleratio=1, ticktext=[f"Br {lv:.3g}%" for lv in levels])
# Reading guide on the first key flower
for x, y, text in [(0.06, 0.45, "bluer"), (0.5, 0.03, "peak"), (0.94, 0.45, "redder")]:
    fig_key.add_annotation(x=x, y=y, text=text, showarrow=False, font={"size": 10, "color": "#6b7280"})
fig_key.update_layout(
    title="Key: what each knob does to the flower"
          "<br><sup>Change over the knob's full design range. "
          "Leans left = peak gets bluer; leans right = redder.</sup>",
    height=600, width=600, template="plotly_white", plot_bgcolor="white", showlegend=False,
)

show_figures([fig_key, fig_flowers])

,unusual,top_bottom,composition_label
well,,,
H8,0.240,0.447,FA 0.500 | Cs 0.300 | Rb 0.100 | DMA 0.100 || ...
E4,0.214,0.380,FA 0.600 | Cs 0.300 | Rb 0.000 | DMA 0.100 || ...
H10,0.209,0.303,FA 0.575 | Cs 0.300 | Rb 0.100 | DMA 0.025 || ...
F9,0.191,0.280,FA 0.675 | Cs 0.300 | Rb 0.025 | DMA 0.000 || ...
D1,0.182,0.391,FA 0.800 | Cs 0.100 | Rb 0.100 | DMA 0.000 || ...
F8,0.170,0.357,FA 0.575 | Cs 0.300 | Rb 0.025 | DMA 0.100 || ...


In [32]:
# ============================================================
# Challenge 1 — property space: where do the wells sit optically, and which knob moves them?
# ============================================================
# Every point = one well. Coordinates = optical properties (from Bottom PL: low noise).
# Marker shape = Br level.  Color = one A-site knob, one panel per knob (Cs | Rb | DMA).
#
#   Figure 1 (context): peak energy vs FWHM           -> the three Br families
#   Figure 2 (main):    Δ peak energy vs Δ FWHM       -> what Cs, Rb, DMA do (Br average removed)
#   Figure 3:           log Top area vs log Bottom area -> brightness (along the diagonals)
#                                                        and Top/Bottom (across the diagonals)
# Uses: quaternary_optical, quaternary_composition, show_figures

HC = 1239.84                       # photon energy (eV) = 1239.84 / wavelength (nm)


# ---------- Step 1: optical properties of every well ----------
def clean_pl(geometry):
    """PL table (rows = wells, columns = wavelengths) with the flat background removed."""
    table = quaternary_optical[quaternary_optical["geometry"].eq(geometry)].pivot_table(
        index="well", columns="wavelength_nm", values="optical_signal_instrument_units")
    return table.sub(table.quantile(0.05, axis=1), axis=0).clip(lower=0)

bottom, top = clean_pl("Bottom"), clean_pl("Top")
wl = bottom.columns.to_numpy(float)
energy = HC / wl                                            # eV of every column

rows = []
for well in bottom.index:
    b = bottom.loc[well].to_numpy()
    t = top.loc[well].to_numpy()
    near = np.abs(wl - wl[b.argmax()]) <= 50                # ±50 nm around the peak

    peak_eV = np.sum(energy[near] * b[near]) / np.sum(b[near])     # weighted mean energy
    above_half = b >= b.max() / 2                           # points above half maximum
    e_max = energy[b.argmax()]
    low_side = e_max - energy[above_half].min()             # half width, low-energy side
    high_side = energy[above_half].max() - e_max            # half width, high-energy side

    rows.append({
        "well": well,
        "peak_eV": peak_eV,
        "fwhm_meV": 1000 * (low_side + high_side),
        "asymmetry": high_side / low_side,                  # > 1 = wider on the blue side
        "log_bottom": np.log10(b[near].sum()),              # brightness near the peak
        "log_top": np.log10(t[near].sum()),
    })
props = quaternary_composition.merge(pd.DataFrame(rows), on="well")

# Remove the Br effect: each property relative to the average of its own Br level
by_br = props.groupby("Br_design_percent")
props["d_peak_meV"] = 1000 * (props["peak_eV"] - by_br["peak_eV"].transform("mean"))
props["d_fwhm_meV"] = props["fwhm_meV"] - by_br["fwhm_meV"].transform("mean")

# Hover text: well, composition, numbers
props["hover"] = (
    "<b>" + props["well"] + "</b><br>" + props["composition_label"]
    + "<br>peak " + props["peak_eV"].round(3).astype(str) + " eV"
    + " | FWHM " + props["fwhm_meV"].round(0).astype(str) + " meV"
    + " | asymmetry " + props["asymmetry"].round(2).astype(str)
    + "<br>Top/Bottom " + (10 ** (props["log_top"] - props["log_bottom"])).round(2).astype(str)
)


# ---------- Shared plotting pieces ----------
BR_LEVELS = sorted(props["Br_design_percent"].unique())
SYMBOLS = dict(zip(BR_LEVELS, ["circle", "square", "diamond"]))
KNOBS = {"Cs_design_percent": "Cs", "Rb_design_percent": "Rb", "DMA_design_percent": "DMA"}

def knob_panels(x, y, x_title, y_title, title):
    """Same scatter three times; panel colors = Cs, Rb, DMA. Shape = Br level."""
    fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                        subplot_titles=[f"color = {name}  ({props[k].min():g} → {props[k].max():g}%)"
                                        for k, name in KNOBS.items()])
    for col, knob in enumerate(KNOBS, start=1):
        for level, symbol in SYMBOLS.items():
            sub = props[props["Br_design_percent"] == level]
            fig.add_trace(go.Scatter(
                x=sub[x], y=sub[y], mode="markers", text=sub["hover"], hoverinfo="text",
                marker={"symbol": symbol, "size": 10, "color": sub[knob], "colorscale": "Viridis",
                        "cmin": props[knob].min(), "cmax": props[knob].max(),
                        "line": {"color": "black", "width": 0.5}},
                showlegend=False,
            ), row=1, col=col)
    # Gray legend entries for the marker shapes
    for level, symbol in SYMBOLS.items():
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=f"Br {level:.3g}%",
                                 marker={"symbol": symbol, "size": 10, "color": "lightgray",
                                         "line": {"color": "black", "width": 0.5}}))
    fig.update_xaxes(title_text=x_title)
    fig.update_yaxes(title_text=y_title, col=1)
    fig.update_layout(
        title=title + "<br><sup>Shape = Br level. Color: purple = lowest, yellow = highest "
                      "level of that panel's knob. No replicates, so no error bars.</sup>",
        height=500, template="plotly_white", legend={"orientation": "h", "y": -0.2},
        margin={"t": 110},
    )
    return fig


# ---------- Figure 1 (context): the three Br families ----------
fig_context = go.Figure()
for level, symbol in SYMBOLS.items():
    sub = props[props["Br_design_percent"] == level]
    fig_context.add_trace(go.Scatter(
        x=sub["peak_eV"], y=sub["fwhm_meV"], mode="markers", name=f"Br {level:.3g}%",
        marker={"symbol": symbol, "size": 10, "line": {"color": "black", "width": 0.5}},
        text=sub["hover"], hoverinfo="text"))
fig_context.update_layout(
    title="Context: Br sets the color and the width"
          "<br><sup>More Br → bluer emission and broader peak (more halide-mixing disorder)</sup>",
    xaxis_title="Peak energy (eV)", yaxis_title="FWHM (meV)",
    height=450, template="plotly_white", margin={"t": 100},
)


# ---------- Figure 2 (main): what the A-site does ----------
fig_shift = knob_panels(
    "d_peak_meV", "d_fwhm_meV",
    "Δ peak energy (meV)<br>(+ = bluer than its Br average)", "Δ FWHM (meV)",
    "A-site effects: peak shift vs broadening, relative to the Br-level average")
fig_shift.add_hline(y=0, line={"color": "gray", "width": 1}, row="all", col="all")
fig_shift.add_vline(x=0, line={"color": "gray", "width": 1}, row="all", col="all")


# ---------- Figure 3: Top vs Bottom brightness ----------
fig_bright = knob_panels(
    "log_bottom", "log_top", "log10 Bottom PL area", "log10 Top PL area",
    "Brightness: along the diagonals = brighter film; across = Top vs Bottom")
# Diagonal guide lines of constant Top/Bottom ratio
x_line = np.array([props["log_bottom"].min() - 0.1, props["log_bottom"].max() + 0.1])
for ratio in [0.25, 0.5, 1.0]:
    for col in [1, 2, 3]:
        fig_bright.add_trace(go.Scatter(
            x=x_line, y=x_line + np.log10(ratio), mode="lines", hoverinfo="skip", showlegend=False,
            line={"color": "gray", "width": 1, "dash": "dot"}), row=1, col=col)
    fig_bright.add_annotation(x=x_line[0], y=x_line[0] + np.log10(ratio), row=1, col=1,
                              text=f"Top/Bottom = {ratio:g}", showarrow=False,
                              xanchor="left", yanchor="bottom", font={"size": 9, "color": "gray"})

show_figures([fig_context, fig_shift, fig_bright])

In [30]:
# ---------- Main figure: the spectra grid (made in the spectra-grid cell, saved as `fig`) ----------
main_figure = fig

# ---------- Evidence spectrum: the Cs effect fades as Br goes up ----------
# All four wells have Rb = 0 and DMA = 0. Only Cs changes (10% -> 30%), at Br 0% and at Br 33%.
PAIRS = {"Br 0%": ("A1", "E1"), "Br 33%": ("A9", "E9")}

fig_evidence = make_subplots(rows=1, cols=2, subplot_titles=list(PAIRS), shared_yaxes=True)
for col, (title, wells) in enumerate(PAIRS.items(), start=1):
    for well, color in zip(wells, ["#93c5fd", "#1d4ed8"]):          # light = Cs 10%, dark = Cs 30%
        rows = quaternary_optical[quaternary_optical["well"].eq(well)
                                  & quaternary_optical["geometry"].eq("Bottom")].sort_values("wavelength_nm")
        signal = rows["optical_signal_instrument_units"].to_numpy(float)
        signal = np.clip(signal - np.percentile(signal, 5), 0, None)   # remove flat background
        cs = quaternary_composition.set_index("well").loc[well, "Cs_design_percent"]
        fig_evidence.add_trace(go.Scatter(
            x=rows["wavelength_nm"], y=signal / signal.max(), mode="lines",
            name=f"{well}: Cs {cs:g}%", line={"color": color, "width": 2.5},
        ), row=1, col=col)
fig_evidence.update_xaxes(range=[640, 850], title_text="Wavelength (nm)")
fig_evidence.update_yaxes(title_text="Bottom PL (peak = 1)", col=1)
fig_evidence.update_layout(
    title="Evidence: Cs 10 → 30% moves the peak ~16 nm bluer at Br 0%, but only ~3 nm at Br 33%"
          "<br><sup>Rb = 0 and DMA = 0 in all four wells</sup>",
    height=450, template="plotly_white",
)

challenge1_figures = [main_figure, fig_evidence]

In [31]:
# ============================================================
# Challenge 2 — 96 small time × wavelength maps on the plate grid
#               (1) Bottom PL   (2) Top PL   (3) Top − Bottom, only where Top is valid
# ============================================================
# The plate layout is the composition grid:
#   columns 1–12 = FAPbI3 host 0 → 100 %  (3AMP spacer = the rest)
#   rows A–H     = MACl level 0 → 20 %     (nominal design label)
# Each cell = one well:  x = wavelength,  y = time (0 min at the bottom, 380 min at the top).
# One fixed color scale per figure. Gray = no valid spectrum, never shown as zero.
#
# Uses: rp_composition, rp_features, rp_spectra_by_id, rp_wavelength_columns, show_figures

WL_MIN, WL_MAX = 560, 840      # no emission below ~560 nm in this library
LOG_MIN, LOG_MAX = 1.5, 5.0    # color range for PL: 10^1.5 ≈ 30 to 10^5 = 100 000 counts
DIFF_MAX = 0.5                 # color range for Top − Bottom: −0.5 … +0.5 (peak = 1)
MIN_PEAK = 100                 # Top − Bottom only where both peaks are above this (counts);
                               # some reads flagged "valid" peak at ~35 counts = noise. 0 = no floor

# Spectra columns inside the window (every 2nd column = 4 nm steps, enough for small maps)
wl_cols = [c for c in rp_wavelength_columns if WL_MIN <= float(c[3:]) <= WL_MAX][::2]
wl = np.array([float(c[3:]) for c in wl_cols])


# ---------- What goes into one cell ----------
def well_spectra(well, measure):
    """All spectra of one well, in time order (rows = times). Invalid reads become NaN."""
    meta = rp_features[rp_features["well"].eq(well)
                       & rp_features["measure"].eq(measure)].sort_values("time_min")
    signal = rp_spectra_by_id.loc[meta["spectrum_id"], wl_cols].to_numpy(float, copy=True)
    signal[~meta["qc_status"].eq("valid").to_numpy()] = np.nan
    return meta["time_min"].to_numpy(), signal

def log_bottom(well):
    times, signal = well_spectra(well, "Bottom PL")
    return times, np.log10(np.clip(signal, 10 ** LOG_MIN, None))

def log_top(well):
    times, signal = well_spectra(well, "Top PL")
    return times, np.log10(np.clip(signal, 10 ** LOG_MIN, None))

def top_minus_bottom(well):
    """Shape difference: each spectrum is scaled to peak = 1 first, so brightness drops out.
    Rows where Top is invalid, or either peak is below MIN_PEAK, become NaN (gray)."""
    times, top = well_spectra(well, "Top PL")
    _, bottom = well_spectra(well, "Bottom PL")
    top_peak = np.max(top, axis=1, keepdims=True)          # NaN for invalid Top reads
    bottom_peak = np.max(bottom, axis=1, keepdims=True)
    difference = top / top_peak - bottom / bottom_peak
    too_weak = (top_peak < MIN_PEAK) | (bottom_peak < MIN_PEAK)
    difference[too_weak[:, 0]] = np.nan
    return times, difference


# ---------- One function that draws the 96-cell grid ----------
fa_levels = sorted(rp_composition["FAPbI3_percent"].unique())     # plate columns, left -> right
macl_levels = sorted(rp_composition["additive_percent"].unique())  # plate rows, top -> bottom

def rp_grid(cell_values, title, value_name, colorscale, zmin, zmax):
    fig = make_subplots(
        rows=8, cols=12, horizontal_spacing=0.006, vertical_spacing=0.012,
        column_titles=[f"FA {p:.0f}%" for p in fa_levels],
        row_titles=[f"MACl {m:g}%" for m in macl_levels],
    )
    traces, trace_rows, trace_cols = [], [], []
    for well in rp_composition.itertuples():
        times, z = cell_values(well.well)
        traces.append(go.Heatmap(
            x=wl, y=times, z=np.round(z, 2), coloraxis="coloraxis",
            hovertemplate=(f"<b>{well.well}</b>  FA {well.FAPbI3_percent:.0f}%  "
                           f"MACl {well.additive_percent:g}%<br>%{{x:.0f}} nm, %{{y:.0f}} min"
                           f"<br>{value_name} %{{z:.2f}}<extra></extra>"),
        ))
        trace_rows.append("ABCDEFGH".index(well.row) + 1)
        trace_cols.append(int(well.col))
    fig.add_traces(traces, rows=trace_rows, cols=trace_cols)   # one call is much faster than 96

    # Guide line at 760 nm in every cell (right of it = the 760–820 nm band)
    fig.add_vline(x=760, line={"color": "gray", "width": 0.7, "dash": "dot"}, row="all", col="all")

    # Tick labels only in the bottom-left cell, which serves as the scale
    fig.update_xaxes(showticklabels=False, ticks="", showgrid=False, range=[WL_MIN, WL_MAX])
    fig.update_yaxes(showticklabels=False, ticks="", showgrid=False, range=[-10, 390])
    fig.update_xaxes(showticklabels=True, tickvals=[600, 700, 800], tickfont={"size": 8}, row=8, col=1)
    fig.update_yaxes(showticklabels=True, tickvals=[0, 380], tickfont={"size": 8}, row=8, col=1)
    fig.update_annotations(font_size=10)                        # column / row titles

    fig.update_layout(
        title=f"{title}: time × wavelength for all 96 wells"
              f"<br><sup>Each cell: {WL_MIN}–{WL_MAX} nm left → right, 0 → 380 min bottom → top. "
              "One color scale for all. Gray = no valid read. Dotted line = 760 nm.</sup>",
        coloraxis={"colorscale": colorscale, "cmin": zmin, "cmax": zmax,
                   "colorbar": {"title": {"text": value_name}, "x": 1.05, "thickness": 14}},
        height=900, template="plotly_white", plot_bgcolor="#d1d5db",
        margin={"l": 40, "r": 100, "t": 110, "b": 40},
    )
    return fig


# ---------- The three figures ----------
fig_rp_bottom = rp_grid(log_bottom, "Bottom PL", "log10 PL", "Magma", LOG_MIN, LOG_MAX)
fig_rp_top = rp_grid(log_top, "Top PL", "log10 PL", "Magma", LOG_MIN, LOG_MAX)
fig_rp_diff = rp_grid(top_minus_bottom, "Top − Bottom shape (only valid, non-weak Top reads)",
                      "Top − Bottom<br>(peak = 1 each)", "RdBu_r", -DIFF_MAX, DIFF_MAX)
# Red = Top has relatively MORE light at this wavelength, blue = LESS.
# A blue/red pair side by side = Top's band sits at a different wavelength than Bottom's.

show_figures([fig_rp_bottom, fig_rp_top, fig_rp_diff])